# Fine tune AG-NEWS Classification Model to IMDB Dataset With LORA

In [390]:
import torch
import math
import time
import torch.nn as nn
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.nn import CrossEntropyLoss


#### **Dataset Loading**

In [391]:
dataset = load_dataset("AG_NEWS")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [392]:
dataset['train'][0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [393]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
tokenizer

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [394]:
label_list = []

for idx,data in enumerate(dataset['train']):
    label_list.append(dataset['train'][idx]['label'])

num_classes = len(set(label_list))
num_classes

4

In [395]:
ag_news_labels = {0: "world", 1:"sports", 2:"business", 3:"sci/tech"}

#### **Define Tokenizer Function**

In [396]:
def tokenize_text (data):
    return tokenizer(data['text'], padding='max_length', truncation=True, max_length=128)


In [397]:
dataset = dataset.map(tokenize_text, batched=True)


Map: 100%|██████████| 7600/7600 [00:01<00:00, 6537.20 examples/s]


In [398]:
dataset.column_names

{'train': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
 'test': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']}

In [399]:
dataset = dataset.rename_column('label', 'labels')


In [400]:
dataset.set_format(type='torch', columns=['labels', 'input_ids', 'attention_mask'])

In [401]:
dataset_train = dataset['train']
dataset_test = dataset['test']

In [402]:
dataset_train[0]

{'labels': tensor(2),
 'input_ids': tensor([  101,  6250,  1457,   119, 10169,   140,  9598,  4388, 14000,  1103,
          2117,   113, 11336, 27603,   114, 11336, 27603,   118,  6373,   118,
         18275,  1116,   117,  6250,  1715,   112,   188,   173, 11129,  1979,
           165,  1467,  1104, 18737,   118,   172,  5730,  4724,   117,  1132,
          3195,  2448,  1254,   119,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,  

In [403]:
dataset_train[0].keys()

dict_keys(['labels', 'input_ids', 'attention_mask'])

In [404]:
dataset_train[0]

{'labels': tensor(2),
 'input_ids': tensor([  101,  6250,  1457,   119, 10169,   140,  9598,  4388, 14000,  1103,
          2117,   113, 11336, 27603,   114, 11336, 27603,   118,  6373,   118,
         18275,  1116,   117,  6250,  1715,   112,   188,   173, 11129,  1979,
           165,  1467,  1104, 18737,   118,   172,  5730,  4724,   117,  1132,
          3195,  2448,  1254,   119,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,  

In [405]:
dataset_test[0]

{'labels': tensor(2),
 'input_ids': tensor([  101, 11284,  1116,  1111,   157,   151, 12966,  1170,  7430,  1913,
          1116,  4311,  3239,  1120,  6217,  1203,  5727,  1474,  1152,  1132,
           112,  9333,   112,  1170,  7430,  1114, 18178,  6486,  3016,  3467,
         12556, 13830,  1233,   119,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,  

**Split train set**

In [406]:
train_valid_split = dataset_train.train_test_split(test_size=0.1, seed=42)

### **Define Dataloaders**

In [431]:
BATCH_SIZE = 8

In [432]:
train_dataloader = DataLoader(train_valid_split['train'], batch_size=BATCH_SIZE, shuffle=True)
valid_dataloader = DataLoader(train_valid_split['test'], batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

### **Define Text-Embedding**

In [433]:
class TextEmbeddings(nn.Module):
    
    def __init__(self, vocab_size, embed_dims, dropout):
        super().__init__()
        
        self.embed_dim = embed_dims
        self.dropout = nn.Dropout(dropout)
        self.embeddings = nn.Embedding(vocab_size, embed_dims)
    
    
    def forward (self, input_tokens):
        # input_token -> [batch_size, seq_len]
        embed_out = self.embeddings(input_tokens) * math.sqrt(self.embed_dim)
        return self.dropout(embed_out)

### **Positional Embeddings**

In [434]:
class PositionalEmbedding (nn.Module):
    
    def __init__(self, vocab_size, d_model, max_seq_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        positions = torch.arange(0, max_seq_len).unsqueeze(1)
        
        pe = torch.zeros(max_seq_len, d_model).float()
        
        div_term = torch.exp(
            torch.arange(0,d_model,2) * (-math.log(10000)/d_model)
        )
        
        pe[:,0::2] = torch.sin(positions * div_term)
        pe[:,1::2] = torch.cos(positions * div_term)
        
        pe = pe.unsqueeze(0)
        
        self.register_buffer("pe",pe)
        
    def forward(self, text_embeddings):
        
        text_embed_size = text_embeddings.size(1)
        pos_embed = text_embeddings + self.pe[:,0:text_embed_size,:]
        return self.dropout(pos_embed)
        

### **Define Model**

In [435]:
print(torch.cuda.is_available())

True


In [436]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [437]:
class CustomClassificationModel(nn.Module):
    
    def __init__(self, vocab_size, d_model,n_classes ,n_layers, n_heads, dropout, dim_feedforward):
        super().__init__()
        
        self.embeddings = TextEmbeddings(vocab_size,d_model, dropout)
        self.pos_embeddings = PositionalEmbedding(vocab_size,d_model,2000, dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(d_model, dim_feedforward=dim_feedforward, 
                                                   batch_first=True, nhead=n_heads)
        
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer=encoder_layer, 
                                                         num_layers=n_layers)
        
        self.linear_layer = nn.Linear(d_model,n_classes)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input_ids, attention_mask:None):
        # input_ids -> [batch_size, seq_len]
        
        embed_out = self.embeddings(input_ids) 
        pos_out = self.pos_embeddings(embed_out)
        
        src_key_padding_mask = (attention_mask == 0) if attention_mask is not None else None
        
        transformer_out = self.transformer_encoder(pos_out,src_key_padding_mask = src_key_padding_mask)
        
        out = transformer_out.mean(dim=1)
        
        out = self.linear_layer(out)
        
        return out
        

### **Define Model Instance** 

In [438]:
vocab_size = tokenizer.vocab_size
d_model = 128 
num_layers = 2
num_heads = 2
dropout = 0.1
dim_feedforward = 256  

In [439]:
custom_model = CustomClassificationModel(vocab_size, d_model, num_classes, num_layers,
                                        num_heads, dropout, dim_feedforward)

In [440]:
custom_model.to(device)

CustomClassificationModel(
  (embeddings): TextEmbeddings(
    (dropout): Dropout(p=0.1, inplace=False)
    (embeddings): Embedding(28996, 128)
  )
  (pos_embeddings): PositionalEmbedding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (linear_layer): Linear(in_features=128, out

### **Define Model Prediction**

In [441]:
def model_predict (model:CustomClassificationModel, input:str):
    
    with torch.no_grad():
        tokens = tokenizer(input, return_tensors='pt').to(device)
        input_ids = tokens['input_ids']
        attention_mask = tokens['attention_mask']
    
        output = model(input_ids,attention_mask)
        out = torch.argmax(output, dim=-1)
        
        predict_label = ag_news_labels[out.item()]
        
        return predict_label
    

In [442]:
sample = "The United Nations held an emergency meeting after rising tensions between neighboring countries sparked concerns about regional stability."

In [443]:
sample_predict_label = model_predict(custom_model,sample)
sample_predict_label

'world'

### **Define Model Evaluation**

In [444]:
def model_eval (model:CustomClassificationModel, dataloader:DataLoader):
    
    model.eval()
    total_accuracy = 0
    total_count = 0
    
    with torch.no_grad():
        
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']
            labels = batch['labels']
            
            output = model(input_ids,attention_mask)
            
            predict_out = torch.argmax(output,dim=-1)
            
            total_accuracy += (predict_out == labels).sum().item()
            total_count += labels.size(0)

    return total_accuracy/total_count
        

In [445]:
pre_train_model_accuracy = model_eval(custom_model,test_dataloader)

In [446]:
print(f"accuracy of custom model before training: {pre_train_model_accuracy * 100:.2f}%")

accuracy of custom model before training: 25.20%


### **Define Model Training**

##### Model Training Parameters

In [447]:
LR = 1e-4  
criterion = CrossEntropyLoss()
optimizer = AdamW(custom_model.parameters(),lr=LR)

In [448]:
num_epochs = 5
total_steps = num_epochs * len(train_dataloader)
warm_steps = int(total_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(optimizer,warm_steps,total_steps)

In [449]:
def model_train(model:CustomClassificationModel, dataloader:DataLoader, criterion:CrossEntropyLoss,
                optimizer:AdamW, lr_scheduler,n_epochs):
    
    total_accuracy = []
    total_loss = []
    accuracy_old = 0
    
    start_time = time.time()
    
    for epoch in tqdm(range(1, n_epochs+1),  desc="Epochs"):
        
        model.train()
        epoch_loss = 0
        
        for i, batch in enumerate(tqdm(dataloader, desc=f"Epoch {epoch}/{n_epochs}")):
            
            batch = {k: v.to(device) for k, v in batch.items()}
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']
            labels = batch['labels']
            
            optimizer.zero_grad()
            
            output = model(input_ids,attention_mask)
            loss = criterion(output, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1)
            
            optimizer.step()
            lr_scheduler.step()
            
            epoch_loss += loss.item()
            
            if i % 1000 == 0:  # ← print every 50 batches
                print(f"  batch {i}, loss: {loss.item():.4f}")
        
        epoch_loss = epoch_loss / len(dataloader)
        total_loss.append(epoch_loss)
        
        accuracy = model_eval(model, valid_dataloader)
        total_accuracy.append(accuracy)
        print(f"Epoch {epoch}/{n_epochs} - Loss: {epoch_loss} Accuracy: {accuracy}") 

        if accuracy > accuracy_old:
            accuracy_old = accuracy
            torch.save(model.state_dict(), "classification_AgNews.pt")
            print("save model epoch",epoch)
    
    end_time = time.time()
    
    print("training time: ", end_time-start_time)
    return total_accuracy, total_loss
    

In [450]:
model_accuracy, model_loss = model_train(custom_model, train_dataloader, criterion,
                                        optimizer, scheduler, num_epochs)

Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

  batch 0, loss: 1.0773


  batch 1000, loss: 1.3434


Epoch 1/5:   8%|▊         | 1013/13500 [00:17<03:25, 60.66it/s]

  batch 2000, loss: 1.3058


...

  batch 11000, loss: 0.0769


  batch 12000, loss: 0.4723


  batch 13000, loss: 0.1600


Epochs: 100%|██████████| 5/5 [42:50<00:00, 514.03s/it]

Epoch 5/5 - Loss: 0.43492736502340135 Accuracy: 0.8348333333333333
save model epoch 5
training time:  2570.1325471401215
